In [ ]:
CATALOG = "demo_medallion"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG}
""")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}
""")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}
""")

In [ ]:
BRONZE_ORDERS = f"{CATALOG}.{BRONZE_SCHEMA}.orders"

df_bronze_orders = spark.table(BRONZE_ORDERS)

display(df_bronze_orders)

In [ ]:
from pyspark.sql.functions import col

df_silver_orders = (
    df_bronze_orders

    # Remove duplicate orders
    .dropDuplicates(["order_id"])

    # Basic validation
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("amount") > 0)

    # Valid order statuses
    .filter(
        col("order_status").isin(
            "COMPLETED",
            "CANCELLED"
        )
    )
)

display(df_silver_orders)

In [ ]:
from pyspark.sql.functions import col

df_silver_orders = (
    df_bronze_orders

    # Remove duplicate orders
    .dropDuplicates(["order_id"])

    # Basic validation
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("amount") > 0)

    # Valid order statuses
    .filter(
        col("order_status").isin(
            "COMPLETED",
            "CANCELLED"
        )
    )
)

display(df_silver_orders)

In [ ]:
df_silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_ORDERS)

print(f"Silver table updated: {SILVER_ORDERS}")